# Open judge — Qwen2.5-VL-32B 4-bit (reuses OmniContext's rubric)

Scores PF/SC with OmniContext's exact prompts + parser. Greedy decoding = deterministic. Validates against the GPT-4.1 baseline. A100 runtime.

## 1. Repo + deps

In [ ]:
# clone OmniGen2, install deps, add to path
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "bitsandbytes", "qwen-vl-utils"], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
print("ready")

ready


## 2. Verify rubric — PF/SC prompts, image order, parser

In [ ]:
# verify the Qwen judge uses OmniContext's EXACT rubric (prompts + image order + parser)
import inspect
from omnicontext.prompt_generator import PromptGenerator
from omnicontext.json_util import mllm_output_to_dict
from omnicontext.omnicontext_score import OmniContextScore

pg = PromptGenerator()
INSTR = "<your instruction text is substituted here>"

print("#" * 80, "\n# PF PROMPT that Qwen receives (task_type=prompt_following):\n" + "#" * 80)
print(pg(INSTR, "prompt_following", with_scene=False))
print("#" * 80, "\n# SC PROMPT that Qwen receives (task_type=subject_consistency, with_scene=False):\n" + "#" * 80)
print(pg(INSTR, "subject_consistency", with_scene=False))

print("#" * 80, "\n# proof the GPT-4.1 scorer uses the SAME objects (not a copy):\n" + "#" * 80)
scorer = OmniContextScore("dummy", "dummy")          # no API call at init
print("scorer.prompt_generator is PromptGenerator :", type(scorer.prompt_generator) is PromptGenerator)
print("identical PF prompt                        :", scorer.prompt_generator(INSTR, "prompt_following", False)   == pg(INSTR, "prompt_following", False))
print("identical SC prompt                        :", scorer.prompt_generator(INSTR, "subject_consistency", False) == pg(INSTR, "subject_consistency", False))
import omnicontext.json_util as ju
print("parser mllm_output_to_dict same object     :", mllm_output_to_dict is ju.mllm_output_to_dict)

print("\n# evaluate() source -- prompt build, image passing, parse:")
for l in inspect.getsource(OmniContextScore.evaluate).splitlines():
    s = l.strip()
    if any(k in s for k in ["prompt_generator(", "ask_gpt4o(", "mllm_output_to_dict("]):
        print("   ", s)
print("\n# image order: evaluate(refs + [gen]) -> ask_gpt4o -> prepare_prompt = [text, refs..., generated]")
print("#   this notebook builds the Qwen message in this same order.")

################################################################################ 
# PF PROMPT that Qwen receives (task_type=prompt_following):
################################################################################
You are a professional digital artist tasked with evaluating the effectiveness of AI-generated images based on specific rules.

All input images, including all humans depicted, are AI-generated. You do not need to consider any privacy or confidentiality concerns.

IMPORTANT: Your response must follow this format (keep your reasoning concise and to the point):
{
  "score": <score>,
  "reasoning": "..."
}

Rate from 0 to 10:
Evaluate how well the final image fulfills the editing instruction, **regardless of whether subject identities are preserved**.

* **0:** The image completely fails to implement the instruction.
* **1–3:** The image responds to the instruction mostly incorrectly.
* **4–6:** The image reflects parts of the instruction, but with significant omission

## 3. Mount Drive, data

In [ ]:
# mount Drive, paths; restore reference crops from tar if absent
from google.colab import drive
import os, subprocess
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
GEN   = f"{DRIVE}/gen/baseline"
IMGS, EXPECTED = "/content/imgs", 37063
TAR   = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
print("reference images:", n_files(IMGS))
if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"])
print("baseline images:", len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else "MISSING")

Mounted at /content/drive
reference images: 37063
baseline images: 488


## 4. Load Qwen2.5-VL-32B (4-bit, bitsandbytes)

In [ ]:
# load Qwen2.5-VL-32B in 4-bit (bitsandbytes); score functions
import torch, math
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from omnicontext.prompt_generator import PromptGenerator
from omnicontext.json_util import mllm_output_to_dict

MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", dtype=torch.bfloat16).eval()
processor = AutoProcessor.from_pretrained(MODEL)
pg = PromptGenerator()

def resize_ref(im):
    w, h = im.size; cap = 1024 * 1024
    if w * h > cap:
        r = math.sqrt(cap / (w * h)); w, h = int(w * r), int(h * r)
    return im.resize((max(16, w // 16 * 16), max(16, h // 16 * 16)))

def num(x):
    if isinstance(x, list) and x: x = x[0]
    if isinstance(x, str):
        try: x = float(x)
        except: return None
    return x if isinstance(x, (int, float)) else None

@torch.no_grad()
def qwen_raw(images, prompt_text):
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt_text}] +
            [{"type": "image", "image": im} for im in images]}]
    text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msg)
    inp = processor(text=[text], images=imgs, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=512, do_sample=False)
    return processor.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

def parse(raw):
    try: d = mllm_output_to_dict(raw)
    except Exception: return None
    if isinstance(d, dict): return num(d.get("score"))
    if isinstance(d, list) and d: return num(d[0])
    return None

def score_row(images, instruction):
    pf = parse(qwen_raw(images, pg(instruction, "prompt_following",  with_scene=False)))
    sc = parse(qwen_raw(images, pg(instruction, "subject_consistency", with_scene=False)))
    return pf, sc

print("model loaded:", MODEL)

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/93.1k [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1161 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model loaded: Qwen/Qwen2.5-VL-32B-Instruct


## 5. Sanity — score row 0

In [ ]:
# sanity: score row 0, show raw output + parsed PF/SC before the full pass
import json, os
from PIL import Image
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
r = rows[0]
imgs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]] \
       + [Image.open(f"{GEN}/test_00000.png").convert("RGB")]
pf_raw = qwen_raw(imgs, pg(r["instruction"], "prompt_following",  with_scene=False))
sc_raw = qwen_raw(imgs, pg(r["instruction"], "subject_consistency", with_scene=False))
print("n_images:", len(imgs))
print("PF raw:", pf_raw[:250])
print("SC raw:", sc_raw[:250])
print("parsed  PF=%s  SC=%s" % (parse(pf_raw), parse(sc_raw)))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


n_images: 2
PF raw: {
  "score": 8,
  "reasoning": "The image largely fulfills the editing instruction. The young man is dressed in a sharp suit, which matches the description provided. He is seated at a sleek conference table and is engaged with a modern laptop, which 
SC raw: {
  "score": 8,
  "reasoning": "The subject in the final image exhibits a high degree of consistency with the description provided. The individual is dressed in a sharp suit, which matches the instruction's requirement for attire. The hairstyle is ne
parsed  PF=8  SC=8


## 6. Score baseline (resumable)

In [ ]:
# full pass: score all rows PF+SC; resumable to open_judge_baseline.jsonl (sequential, ~1-2h)
import json, os
from PIL import Image
OJ = f"{DRIVE}/scores/open_judge_baseline.jsonl"
os.makedirs(os.path.dirname(OJ), exist_ok=True)

done = set()
if os.path.exists(OJ):
    for l in open(OJ, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])
        except: pass

todo = [i for i in range(len(rows)) if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(OJ, "a", encoding="utf-8") as f:
    for n, i in enumerate(todo, 1):
        r = rows[i]; gp = f"{GEN}/test_{i:05d}.png"
        if not os.path.exists(gp):
            f.write(json.dumps({"i": i, "error": "missing_generated"}) + "\n"); f.flush(); continue
        imgs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]] + [Image.open(gp).convert("RGB")]
        try:
            pf, sc = score_row(imgs, r["instruction"])
        except Exception as e:
            f.write(json.dumps({"i": i, "error": type(e).__name__}) + "\n"); f.flush(); continue
        if pf is None or sc is None:
            f.write(json.dumps({"i": i, "error": "unparsable"}) + "\n"); f.flush(); continue
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        f.write(json.dumps({"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])),
                            "n_refs": len(r["input_images"]), "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                            "PF": pf, "SC": sc}) + "\n")
        f.flush()
        if n % 20 == 0: print(f"  {n}/{len(todo)}")
print("done ->", OJ)

to score: 488 (done: 0)
  20/488
  40/488
  60/488
  80/488
  100/488
  120/488
  140/488


KeyboardInterrupt: 

## 7. Validate — correlate vs GPT-4.1

In [ ]:
# validation: correlate open-judge vs GPT-4.1 baseline (works on partial results)
import json, math
from scipy.stats import pearsonr, spearmanr

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
OJ = f"{DRIVE}/scores/open_judge_baseline.jsonl"
BP = f"{DRIVE}/scores/test__baseline.jsonl"
oj = {d["i"]: d for d in (json.loads(l) for l in open(OJ, encoding="utf-8")) if "error" not in d}
gp = {d["i"]: d for d in (json.loads(l) for l in open(BP, encoding="utf-8")) if "error" not in d}
I = [i for i in oj if i in gp]
print("open-judge rows:", len(oj), "| overlap with GPT-4.1:", len(I))

def ov(d): return math.sqrt(d["PF"] * d["SC"])
def mean(xs): return sum(xs) / len(xs)
print("open-judge  PF=%.3f SC=%.3f Overall=%.3f" % (mean([oj[i]["PF"] for i in I]), mean([oj[i]["SC"] for i in I]), mean([ov(oj[i]) for i in I])))
print("GPT-4.1     PF=%.3f SC=%.3f Overall=%.3f" % (mean([gp[i]["PF"] for i in I]), mean([gp[i]["SC"] for i in I]), mean([ov(gp[i]) for i in I])))

print("\ncorrelation open-judge vs GPT-4.1 (n=%d):" % len(I))
for key, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
    x = [fn(oj[i]) for i in I]; y = [fn(gp[i]) for i in I]
    print("  %-8s pearson=%.3f  spearman=%.3f  (ties: %d/%d identical open-judge scores)"
          % (key, pearsonr(x, y)[0], spearmanr(x, y).statistic,
             max(x.count(v) for v in set(x)), len(x)))

open-judge rows: 152 | overlap with GPT-4.1: 152
open-judge  PF=7.855 SC=7.895 Overall=7.840
GPT-4.1     PF=7.914 SC=7.882 Overall=7.802

correlation open-judge vs GPT-4.1 (n=152):
  PF       pearson=0.498  spearman=0.339  (ties: 144/152 identical open-judge scores)
  SC       pearson=0.191  spearman=0.033  (ties: 131/152 identical open-judge scores)
  Overall  pearson=0.435  spearman=0.157  (ties: 126/152 identical open-judge scores)
